In [ ]:
from joblib import load

from tqdm.auto import tqdm

from sklearn.model_selection import (
    train_test_split,
    KFold
)

import torch

import tensorflow as tf

from joblib import load

from keras.utils import to_categorical

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [ ]:
# convert all sublists to arrays
def flatten(subject):
    array_data = [np.array(x) for x in subject]  
    flattened_data = np.concatenate(array_data, axis=0)
    return flattened_data

## GPU prepareation

In [ ]:
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, Subset

# PyTorch Geometric
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader



class NVGDataset(Dataset):

    def __init__(self, X, A, y):

        self.X = np.asarray(X, dtype=np.float32)
        self.A = np.asarray(A, dtype=np.uint8)
        self.y = np.asarray(y, dtype=np.float32)

        assert self.X.shape[1:] == (25, 1)
        assert self.A.shape[1:] == (25, 25)
        assert len(self.X) == len(self.A) == len(self.y)

        self.T = self.X.shape[1]

        self.time = np.linspace(
            0, 1, self.T, dtype=np.float32
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        values = self.X[idx, :, 0]
        A = self.A[idx]


        A_with_self = A.copy()
        np.fill_diagonal(A_with_self, 1)

        degree = (
            (A_with_self > 0).sum(axis=1)
            / self.T
        ).astype(np.float32)

 
        node_features = np.stack(
            [
                values,
                self.time,
                degree
            ],
            axis=1
        ).astype(np.float32)

        A_edges = A.copy()
        np.fill_diagonal(A_edges, 0)

        src, dst = np.nonzero(A_edges)

        edge_index = torch.tensor(
            np.stack([src, dst]),
            dtype=torch.long
        )

        dy = values[src] - values[dst]
        dt = self.time[src] - self.time[dst]

        distances = np.sqrt(
            dy ** 2 + dt ** 2
        ).astype(np.float32)

        edge_attr = torch.from_numpy(
            distances[:, None]
        )

        label = int(np.argmax(self.y[idx]))

        return Data(
            x=torch.from_numpy(node_features),
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor(label, dtype=torch.long)
        )


def create_nvg_dataset(
    X,
    y,
    batch_size=2048,
    shuffle=True,
    rand_seed = 42
):


    if isinstance(X, torch.Tensor):
        X_np = X.detach().cpu().numpy()
    else:
        X_np = np.asarray(X)

    X_np = X_np.astype(np.float32, copy=False)

    if isinstance(y, torch.Tensor):
        y_np = y.detach().cpu().numpy()
    else:
        y_np = np.asarray(y)

    N, T, n_features = X_np.shape

    assert T == 25
    assert n_features == 1
    assert len(y_np) == N



    A_nvg = np.zeros(
        (N, T, T),
        dtype=np.uint8
    )

    for i in tqdm(range(N), desc="Building NVGs"):

        ts = X_np[i, :, 0]

        vg = NaturalVG()
        vg.build(ts)

        edges = np.asarray(
            vg.edges,
            dtype=np.int64
        )

        if edges.size > 0:

            src = edges[:, 0]
            dst = edges[:, 1]

            A_nvg[i, src, dst] = 1
            A_nvg[i, dst, src] = 1


    dataset = NVGDataset(
        X=X_np,
        A=A_nvg,
        y=y_np
    )

    generator = torch.Generator()
    generator.manual_seed(rand_seed)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        generator=generator
    )

    print("X:", X_np.shape)
    print("NVG adjacency:", A_nvg.shape)
    print("Number of graphs:", len(dataset))

    return dataset, loader, A_nvg


def save_packed_nvg(A_nvg, count, split):

    save_dir = Path("/GAT/data/")
    save_dir.mkdir(parents=True, exist_ok=True)

    file_path = save_dir / f"A_nvg_{split}_fold_{count}.npy"

    N, T, _ = A_nvg.shape
    n_bits = T * T
    n_bytes = (n_bits + 7) // 8

    packed = np.lib.format.open_memmap(
        file_path,
        mode="w+",
        dtype=np.uint8,
        shape=(N, n_bytes)
    )

    for start in range(0, N, 10000):
        end = min(start + 10000, N)

        chunk = A_nvg[start:end].reshape(-1, n_bits)

        packed[start:end] = np.packbits(
            chunk,
            axis=1
        )

    packed.flush()
    del packed

    print("Saved to:", file_path.resolve())
    print("File size (GB):", file_path.stat().st_size / 1e9)

    return file_path


In [ ]:
import numpy as np
from ts2vg import NaturalVG

# X_train_t shape:
# (630399, 25, 1)

import os
import random
import numpy as np
import torch

# Ensures reproducibility
seed_value = 42

os.environ["PYTHONHASHSEED"] = str(seed_value)

random.seed(seed_value)
np.random.seed(seed_value)

torch.manual_seed(seed_value)


BATCH_SIZE = 2048 
SEED = 42


X_data, Y_data = load("/Data/3Fold_Train_Data.joblib")

# initializes 5 folds for cross validation
kfolds = KFold(n_splits=3, shuffle=False)#, random_state=42)

count = 0

for train_idx, test_idx in kfolds.split(X_data):

    # split into temp and test (20%)
    train_idx, val_idx = train_test_split(train_idx, test_size=0.05, shuffle=False, random_state=seed_value)
    print(len(train_idx))
    print(len(val_idx))
    print(len(test_idx))

    average_precisions = []
    PRauc_scores = []

    print(f"Started for fold {count}")

    # extracts the selected subjects for training for the X and Y folds
    X_train_fold = [X_data[i] for i in train_idx]
    Y_train_fold = [Y_data[i] for i in train_idx]
    X_train_fold  = flatten(X_train_fold)
    Y_train_fold  = flatten(Y_train_fold)

    # extracts the selected subjects for validating for the X and Y folds
    X_val_fold = [X_data[i] for i in val_idx]
    Y_val_fold = [Y_data[i] for i in val_idx]
    X_val_fold  = flatten(X_val_fold)
    Y_val_fold  = flatten(Y_val_fold)

    X_test_fold = [X_data[i] for i in test_idx]
    Y_test_fold = [Y_data[i] for i in test_idx]
    # flattens the arrays of single subjects into one array of shape (X,25,2)
    X_test_fold = flatten(X_test_fold)
    # flattens the arrays of single subjects into one array of shape (X,1)
    Y_test_fold = flatten(Y_test_fold)

    Y_test_rs = to_categorical(Y_test_fold)
    X_test_t = tf.convert_to_tensor(X_test_fold, dtype=tf.float32)
    Y_test_t = tf.convert_to_tensor(Y_test_rs, dtype=tf.int32)
    Y_test_all = Y_test_fold.reshape(-1)

    _, Train_loader, Train_nvg = create_nvg_dataset(
        X_train_fold, Y_train_fold,
        batch_size=2048,
        shuffle=True,
        rand_seed = 42
    ) 

    _, Val_loader, Val_nvg = create_nvg_dataset(
        X_val_fold, Y_val_fold,
        batch_size=2048,
        shuffle=False,
        rand_seed = 42
    )
    print("done")

    _, Test_loader, Test_nvg = create_nvg_dataset(
        X_test_fold, Y_test_fold,
        batch_size=2048,
        shuffle=False,
        rand_seed = 42
    )

    save_packed_nvg(Train_nvg, count, "train")
    save_packed_nvg(Val_nvg, count, "val")
    save_packed_nvg(Test_nvg, count, "test")

    np.save(f"/GAT/data/X_train_fold_{count}.npy", X_train_fold)
    np.save(f"/GAT/data/X_val_fold_{count}.npy", X_val_fold)
    np.save(f"/GAT/data/X_test_fold_{count}.npy", X_test_fold)

    np.save(f"/GAT/data/Y_train_fold_{count}.npy", Y_train_fold)
    np.save(f"/GAT/data/Y_val_fold_{count}.npy", Y_val_fold)
    np.save(f"/GAT/data/Y_test_fold_{count}.npy", Y_test_fold)

    count +=1

In [ ]:
import numpy as np
from ts2vg import NaturalVG

# X_train_t shape:
# (630399, 25, 1)

import os
import random
import numpy as np
import torch

# Ensures reproducibility
seed_value = 42

os.environ["PYTHONHASHSEED"] = str(seed_value)

random.seed(seed_value)
np.random.seed(seed_value)

torch.manual_seed(seed_value)


BATCH_SIZE = 2048 
SEED = 42


X_data, Y_data = load("/Data/3Fold_Train_Data.joblib")



X_train_fold  = flatten(X_data)
Y_train_fold  = flatten(Y_data)

_, Train_loader, Train_nvg = create_nvg_dataset(
    X_train_fold, Y_train_fold,
    batch_size=2048,
    shuffle=True,
    rand_seed = 42
) 

save_packed_nvg(Train_nvg, count, "Full_Train")

In [ ]:
import numpy as np
from ts2vg import NaturalVG

# X_train_t shape:
# (630399, 25, 1)

import os
import random
import numpy as np
import torch

# Ensures reproducibility
seed_value = 42

os.environ["PYTHONHASHSEED"] = str(seed_value)

random.seed(seed_value)
np.random.seed(seed_value)

torch.manual_seed(seed_value)


BATCH_SIZE = 2048 
SEED = 42
count = 0

X_DL_test, Y_test = load("Data/DL_Test_True.joblib")


X_DL_test = flatten(X_DL_test)
Y_test  = flatten(Y_test)

_, Test_DL_loader, DL_nvg = create_nvg_dataset(
    X_DL_test, Y_test,
    batch_size=2048,
    shuffle=True,
    rand_seed = 42
) 

save_packed_nvg(DL_nvg, count, "OS_Test")